In [1]:
import pandas as pd
import numpy as np
import psycopg
from psycopg import sql
import os
import sys
from dotenv import load_dotenv
import matplotlib.pyplot as plt
from datetime import datetime, timedelta, timezone
import time

sys.path.append(os.path.abspath('./src'))
import db_functions as dbf
load_dotenv()
user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
host = os.getenv("DB_HOST", "localhost")
port = os.getenv("DB_PORT", "5432")
dbname = os.getenv("DB_NAME")
conn_str = f"postgresql://{user}:{password}@{host}:{port}/{dbname}"
api_key = os.getenv("API_KEY")
api_url = 'https://api.stratz.com/graphql'
headers = {
    'User-Agent': 'STRATZ_API',
    "Authorization": f"Bearer {api_key}"
}

In [2]:
def get_all_stratz_weeks(start_datetime):
    """
    Returns a list of Unix timestamps for every Sunday at 00:00:00 UTC 
    from the start_datetime until now.
    """
    # 1. Ensure we are working in UTC and at midnight
    start_dt = start_datetime.replace(hour=0, minute=0, second=0, microsecond=0)
    if start_dt.tzinfo is None:
        start_dt = start_dt.replace(tzinfo=timezone.utc)
    
    # 2. "Snap" to the Sunday of that week (Stratz weeks start on Sunday)
    # weekday(): 0=Mon, 6=Sun. To get to Sunday, we subtract (weekday + 1) % 7
    days_to_subtract = (start_dt.weekday() + 1) % 7
    current_week_dt = start_dt - timedelta(days=days_to_subtract)
    
    now = datetime.now(timezone.utc)
    week_timestamps = []
    
    # 3. Iterate forward until we hit the current time
    while current_week_dt <= now:
        week_timestamps.append(int(current_week_dt.timestamp()))
        current_week_dt += timedelta(weeks=1)
        
    return week_timestamps

In [ ]:
query = '''
    query {
        constants {
            heroes(gameVersionId: 182) { 
                id
            }
        }
    }
''' #TODO: replace hard-coded value from database
results = dbf.query_stratz(query, headers, api_url)
hero_ids = [res['id'] for res in results['data']['constants']['heroes']]
weeks = get_all_stratz_weeks(datetime(2025, 2, 21))
weeks = weeks[-15:] #Stratz only keeps last 15 weeks' data

In [ ]:
query = '''
    query($heroIds: [Short]!, $week: Long, $bracketBasicIds: [RankBracketBasicEnum]) {
        heroStats {
            stats(heroIds: $heroIds, week: $week, bracketBasicIds: $bracketBasicIds) {
            heroId
            week
            time
            position
            bracketBasicIds
            matchCount
            winCount
            networth
            goldPerMinute
            towerDamage
            disableDuration
            disableCount
            stunDuration
            stunCount
            healingSelf
            healingAllies
            heroDamage
            physicalDamage
            magicalDamage
            physicalDamageReceived
            magicalDamageReceived
            supportGold
            campsStacked
            }
        }
    }
'''
for week in weeks:
    with psycopg.connect(conn_str) as conn:
        with conn.cursor() as cur:
            variables = {'heroIds': hero_ids, 'week': week, 'bracketBasicIds': 'DIVINE_IMMORTAL'}
            result = dbf.query_stratz(query, headers, api_url=api_url, variables=variables)
            res = result['data']['heroStats']['stats']
            df = pd.DataFrame(res)
            if week == weeks[-15]:
                dbf.create_table_from_df(df, 'hero_stats', conn_str, convert_dtypes=False, add_serial_id=True)
            dbf.insert_df_into_table(df, 'hero_stats', conn_str)
            # After the data is inserted, sync the sequence
            seq_query = f"SELECT setval(pg_get_serial_sequence('\"hero_stats\"', 'id'), max(id)) FROM \"hero_stats\";"
            cur.execute(seq_query)


Table 'hero_stats' created successfully.
Data inserted into table 'hero_stats' successfully.
Data inserted into table 'hero_stats' successfully.
Data inserted into table 'hero_stats' successfully.
Data inserted into table 'hero_stats' successfully.
Data inserted into table 'hero_stats' successfully.
Data inserted into table 'hero_stats' successfully.
Data inserted into table 'hero_stats' successfully.
Data inserted into table 'hero_stats' successfully.
Data inserted into table 'hero_stats' successfully.
Data inserted into table 'hero_stats' successfully.
Data inserted into table 'hero_stats' successfully.
Data inserted into table 'hero_stats' successfully.
Data inserted into table 'hero_stats' successfully.
Data inserted into table 'hero_stats' successfully.
Data inserted into table 'hero_stats' successfully.


In [19]:
query = '''
    query($heroIds: [Short]!, $week: Long, $bracketBasicIds: [RankBracketBasicEnum]) {
        heroStats {
            matchUp(heroIds: $heroIds, week: $week, bracketBasicIds: $bracketBasicIds) {
            heroId
            matchCountWith
            matchCountVs
            with {
                heroId1
                heroId2
                week
                synergy
                winCount
                matchCount
                winsAverage
                goldEarned
                xp
                heroDamage
                towerDamage
                firstBloodTime
                synergy
                winRateHeroId1
                winRateHeroId2
            }
            vs {
                heroId1
                heroId2
                week
                synergy
                winCount
                matchCount
                winsAverage
                goldEarned
                xp
                heroDamage
                towerDamage
                firstBloodTime
                synergy
                winRateHeroId1
                winRateHeroId2
            }
        }
    }
}
'''
for week in weeks:
    variables = {'heroIds': hero_ids, 'week': week, 'bracketBasicIds': 'DIVINE_IMMORTAL'}
    result = dbf.query_stratz(query, headers, api_url=api_url, variables=variables)
    res = result['data']['heroStats']['matchUp']
    info = {}
    df_main_stats = pd.DataFrame(columns=['heroId', 'week', 'matchCountWith', 'matchCountVs'])
    df_with_stats = pd.DataFrame(
        columns=['heroId1', 'heroId2', 'week', 'synergy', 'winCount', 'matchCount',
       'winsAverage', 'goldEarned', 'xp', 'heroDamage', 'towerDamage',
       'firstBloodTime', 'winRateHeroId1', 'winRateHeroId2']
    )
    df_vs_stats = pd.DataFrame(
        columns=['heroId1', 'heroId2', 'week', 'synergy', 'winCount', 'matchCount',
       'winsAverage', 'goldEarned', 'xp', 'heroDamage', 'towerDamage',
       'firstBloodTime', 'winRateHeroId1', 'winRateHeroId2']
    )
    for hero in res:
        try:
            info['heroId'] = hero['heroId']
            info['week'] = hero['with'][0]['week'] 
            info['matchCountWith'] = hero['matchCountWith']
            info['matchCountVs'] = hero['matchCountVs']
            series = pd.Series(info)
            df_main_stats = pd.concat([df_main_stats, series.to_frame().transpose()])
            df_with_stats = pd.concat([df_with_stats, pd.DataFrame(hero['with'])])
            df_vs_stats = pd.concat([df_with_stats, pd.DataFrame(hero['vs'])])
        except:
            continue
    if week == weeks[-15]:
        dbf.create_table_from_df(df_main_stats, 'matchup_stats', conn_str, add_serial_id=True)
        dbf.create_table_from_df(df_with_stats, 'matchup_with', conn_str, add_serial_id=True)
        dbf.create_table_from_df(df_vs_stats, 'matchup_vs', conn_str, add_serial_id=True)
    dbf.insert_df_into_table(df_main_stats, 'matchup_stats', conn_str)
    dbf.insert_df_into_table(df_with_stats, 'matchup_with', conn_str)
    dbf.insert_df_into_table(df_vs_stats, 'matchup_vs', conn_str)

C:\Users\benib\AppData\Local\Temp\ipykernel_31108\3354295094.py:70: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_with_stats = pd.concat([df_with_stats, pd.DataFrame(hero['with'])])


Table 'matchup_stats' created successfully.
Table 'matchup_with' created successfully.
Table 'matchup_vs' created successfully.
Data inserted into table 'matchup_stats' successfully.
Data inserted into table 'matchup_with' successfully.
Data inserted into table 'matchup_vs' successfully.


C:\Users\benib\AppData\Local\Temp\ipykernel_31108\3354295094.py:70: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_with_stats = pd.concat([df_with_stats, pd.DataFrame(hero['with'])])


Data inserted into table 'matchup_stats' successfully.
Data inserted into table 'matchup_with' successfully.
Data inserted into table 'matchup_vs' successfully.


C:\Users\benib\AppData\Local\Temp\ipykernel_31108\3354295094.py:70: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_with_stats = pd.concat([df_with_stats, pd.DataFrame(hero['with'])])


...

Data inserted into table 'matchup_stats' successfully.
Data inserted into table 'matchup_with' successfully.
Data inserted into table 'matchup_vs' successfully.


C:\Users\benib\AppData\Local\Temp\ipykernel_31108\3354295094.py:70: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_with_stats = pd.concat([df_with_stats, pd.DataFrame(hero['with'])])


Data inserted into table 'matchup_stats' successfully.
Data inserted into table 'matchup_with' successfully.
Data inserted into table 'matchup_vs' successfully.


C:\Users\benib\AppData\Local\Temp\ipykernel_31108\3354295094.py:70: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_with_stats = pd.concat([df_with_stats, pd.DataFrame(hero['with'])])


Data inserted into table 'matchup_stats' successfully.
Data inserted into table 'matchup_with' successfully.
Data inserted into table 'matchup_vs' successfully.


In [6]:
query = '''
    query($heroId: Short!, $week: Long, $bracketBasicIds: [RankBracketBasicEnum]) {
        heroStats {
            itemFullPurchase(heroId: $heroId, week: $week, bracketBasicIds: $bracketBasicIds) {
            heroId
            week
            itemId
            instance
            time
            matchCount
            winCount
            winsAverage
            }
            itemStartingPurchase(heroId: $heroId, week: $week) {
            heroId
            week
            itemId
            instance
            wasGiven
            matchCount
            winCount
            winsAverage
            } 
            talent(heroId: $heroId, week: $week, bracketBasicIds: $bracketBasicIds) {
            heroId
            week
            abilityId
            matchCount
            winCount
            time
            winsAverage
            timeAverage
            }
            abilityMinLevel(heroId: $heroId, week: $week, bracketBasicIds: $bracketBasicIds) {
            heroId
            week
            abilityId
            level
            matchCount
            winCount
            }
            abilityMaxLevel(heroId: $heroId, week: $week, bracketBasicIds: $bracketBasicIds) {
            heroId
            week
            abilityId
            level
            matchCount
            winCount
            }
        }
    }
'''
## Weeks 0-5 done
for week in weeks[12:]:
    df_item_full_purchase = pd.DataFrame(
        columns=[
            'heroId',
            'week',
            'itemId',
            'instance',
            'time',
            'matchCount',
            'winCount',
            'winsAverage'
        ]
    )
    df_item_starting_purchase = pd.DataFrame(
        columns=[
            'heroId',
            'week',
            'itemId',
            'instance',
            'wasGiven',
            'matchCount',
            'winCount',
            'winsAverage'
        ]
    )
    df_talent = pd.DataFrame(
        columns=[
            'heroId',
            'week',
            'abilityId',
            'matchCount',
            'winCount',
            'time',
            'winsAverage',
            'timeAverage'
        ]
    )
    ability_cols = [
        'heroId',
        'week',
        'abilityId',
        'level',
        'matchCount',
        'winCount'
    ]
    df_ability_min_level = pd.DataFrame(columns=ability_cols)
    df_ability_max_level = pd.DataFrame(columns=ability_cols)
    for idx, hero_id in enumerate(hero_ids):
        start_time = time.time()
        variables = {'heroId': hero_id, 'week': week, 'bracketBasicIds': 'DIVINE_IMMORTAL'}
        result = dbf.query_stratz(query, headers, api_url=api_url, variables=variables)
        res = result['data']['heroStats']
        df_item_full_purchase = pd.concat([df_item_full_purchase, pd.DataFrame(res['itemFullPurchase'])])
        df_item_starting_purchase = pd.concat([df_item_starting_purchase, pd.DataFrame(res['itemStartingPurchase'])])
        df_talent = pd.concat([df_talent, pd.DataFrame(res['talent'])])
        df_ability_min_level = pd.concat([df_ability_min_level, pd.DataFrame(res['abilityMinLevel'])])
        df_ability_max_level = pd.concat([df_ability_max_level, pd.DataFrame(res['abilityMaxLevel'])])
    if week == weeks[0]:
        dbf.create_table_from_df(df_item_full_purchase, 'hero_item_full_purchase', conn_str, add_serial_id=True)
        dbf.create_table_from_df(df_item_starting_purchase, 'hero_item_starting_purchase', conn_str, add_serial_id=True)
        dbf.create_table_from_df(df_talent, 'hero_talent', conn_str, add_serial_id=True)
        dbf.create_table_from_df(df_ability_min_level, 'hero_ability_min', conn_str, add_serial_id=True)
        dbf.create_table_from_df(df_ability_max_level, 'hero_ability_max', conn_str, add_serial_id=True)
    dbf.insert_df_into_table(df_item_full_purchase, 'hero_item_full_purchase', conn_str)
    dbf.insert_df_into_table(df_item_starting_purchase, 'hero_item_starting_purchase', conn_str)
    dbf.insert_df_into_table(df_talent, 'hero_talent', conn_str)
    dbf.insert_df_into_table(df_ability_min_level, 'hero_ability_min', conn_str)
    dbf.insert_df_into_table(df_ability_max_level, 'hero_ability_max', conn_str)
            
        

C:\Users\benib\AppData\Local\Temp\ipykernel_12020\3846830479.py:106: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_item_full_purchase = pd.concat([df_item_full_purchase, pd.DataFrame(res['itemFullPurchase'])])
C:\Users\benib\AppData\Local\Temp\ipykernel_12020\3846830479.py:107: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_item_starting_purchase = pd.concat([df_item_starting_purchase, pd.DataFrame(res['itemStartingPurchase'])])
C:\Users\benib\AppData\Local\Temp\ipykernel_12020\3846830479.p

Data inserted into table 'hero_item_full_purchase' successfully.
Data inserted into table 'hero_item_starting_purchase' successfully.
Data inserted into table 'hero_talent' successfully.
Data inserted into table 'hero_ability_min' successfully.
Data inserted into table 'hero_ability_max' successfully.


C:\Users\benib\AppData\Local\Temp\ipykernel_12020\3846830479.py:106: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_item_full_purchase = pd.concat([df_item_full_purchase, pd.DataFrame(res['itemFullPurchase'])])
C:\Users\benib\AppData\Local\Temp\ipykernel_12020\3846830479.py:107: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_item_starting_purchase = pd.concat([df_item_starting_purchase, pd.DataFrame(res['itemStartingPurchase'])])
C:\Users\benib\AppData\Local\Temp\ipykernel_12020\3846830479.p

Data inserted into table 'hero_item_full_purchase' successfully.
Data inserted into table 'hero_item_starting_purchase' successfully.
Data inserted into table 'hero_talent' successfully.
Data inserted into table 'hero_ability_min' successfully.
Data inserted into table 'hero_ability_max' successfully.


C:\Users\benib\AppData\Local\Temp\ipykernel_12020\3846830479.py:106: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_item_full_purchase = pd.concat([df_item_full_purchase, pd.DataFrame(res['itemFullPurchase'])])
C:\Users\benib\AppData\Local\Temp\ipykernel_12020\3846830479.py:107: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_item_starting_purchase = pd.concat([df_item_starting_purchase, pd.DataFrame(res['itemStartingPurchase'])])
C:\Users\benib\AppData\Local\Temp\ipykernel_12020\3846830479.p

Data inserted into table 'hero_item_full_purchase' successfully.
Data inserted into table 'hero_item_starting_purchase' successfully.
Data inserted into table 'hero_talent' successfully.
Data inserted into table 'hero_ability_min' successfully.
Data inserted into table 'hero_ability_max' successfully.


In [15]:
query = '''
    query($heroId: Short, $week: Long, $bracketBasicIds: [RankBracketBasicEnum], $isWith: Boolean!) {
        heroStats {
            laneOutcome(heroId: $heroId, week: $week, bracketBasicIds: $bracketBasicIds, isWith: $isWith) {
                heroId1
                heroId2
                week
                matchCount
                drawCount
                winCount
                lossCount
                stompWinCount
                stompLossCount
                matchWinCount
                csCount
            }
        }
    }
'''
#Week 0-14 done
for week in weeks[14:]:
    if week == 1:
        break
    start_time = time.time()
    df_lane_outcome = pd.DataFrame(
        columns=[
            'heroId1',
            'heroId2',
            'week',
            'matchCount',
            'drawCount',
            'winCount',
            'lossCount',
            'stompWinCount',
            'stompLossCount',
            'matchWinCount',
            'csCount'
        ]
    )
    for is_with in [True, False]:
        for idx, hero_id in enumerate(hero_ids):
            variables = {'heroId': hero_id, 'week': week, 'bracketBasicIds': 'DIVINE_IMMORTAL', 'isWith': is_with}
            result = dbf.query_stratz(query, headers, api_url=api_url, variables=variables)
            res = result['data']['heroStats']['laneOutcome']
            df_lane_outcome = pd.concat([df_lane_outcome, pd.DataFrame(res)])
    if week == weeks[0]:
        dbf.create_table_from_df(df_lane_outcome, 'matchup_lane_outcome', conn_str, add_serial_id=True)
    dbf.insert_df_into_table(df_lane_outcome, 'matchup_lane_outcome', conn_str)
    elapsed = time.time() - start_time
    if elapsed < 0.2:
        time.sleep(0.2-elapsed)


Data inserted into table 'matchup_lane_outcome' successfully.
